In [2]:
# STEP 1: IMPORT LIBRARY

import sqlite3
import pandas as pd

# STEP 2: CONNECT TO DATABASE

conn = sqlite3.connect("academic_records.db")
cursor = conn.cursor()

# STEP 3: CREATE STUDENTS TABLE

cursor.execute("""
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY,
    student_name TEXT NOT NULL,
    department TEXT,
    admission_year INTEGER
)
""")

# STEP 4: CREATE MARKS TABLE

cursor.execute("""
CREATE TABLE IF NOT EXISTS marks (
    mark_id INTEGER PRIMARY KEY,
    student_id INTEGER,
    subject TEXT,
    marks_obtained INTEGER,
    FOREIGN KEY (student_id) REFERENCES students(student_id)
)
""")

# STEP 5: INSERT 10 STUDENTS

students_data = [
    (1, 'Arun', 'CSE', 2020),
    (2, 'Priya', 'CSE', 2021),
    (3, 'Kumar', 'ECE', 2022),
    (4, 'Divya', 'ECE', 2023),
    (5, 'Ravi', 'IT', 2020),
    (6, 'Meena', 'IT', 2021),
    (7, 'Anu', 'CSE', 2024),
    (8, 'Vijay', 'ECE', 2019),
    (9, 'Kavi', 'IT', 2022),
    (10, 'Siva', 'CSE', 2023)
]

cursor.executemany("""
INSERT OR IGNORE INTO students
(student_id, student_name, department, admission_year)
VALUES (?, ?, ?, ?)
""", students_data)

# STEP 6: INSERT MARKS

marks_data = [
    (1, 1, 'Python', 80),
    (2, 1, 'SQL', 70),
    (3, 1, 'Maths', 85),
    (4, 2, 'Python', 90),
    (5, 2, 'SQL', 82),
    (6, 2, 'Maths', 88),
    (7, 3, 'Python', 60),
    (8, 3, 'SQL', 65),
    (9, 3, 'Maths', 70),
    (10, 4, 'Python', 78),
    (11, 4, 'SQL', 80),
    (12, 4, 'Maths', 75),
    (13, 5, 'Python', 40),
    (14, 5, 'SQL', 30),
    (15, 5, 'Maths', 25),
    (16, 6, 'Python', 85),
    (17, 6, 'SQL', 90),
    (18, 6, 'Maths', 88),
    (19, 7, 'Python', 95),
    (20, 7, 'SQL', 92),
    (21, 7, 'Maths', 96),
    (22, 8, 'Python', 32),
    (23, 8, 'SQL', 30),
    (24, 8, 'Maths', 28),
    (25, 9, 'Python', 68),
    (26, 9, 'SQL', 72),
    (27, 9, 'Maths', 75),
    (28, 10, 'Python', 80),
    (29, 10, 'SQL', 78),
    (30, 10, 'Maths', 82)
]

cursor.executemany("""
INSERT OR IGNORE INTO marks
(mark_id, student_id, subject, marks_obtained)
VALUES (?, ?, ?, ?)
""", marks_data)

conn.commit()

# STEP 7: UPDATE MARKS BY +5 FOR STUDENTS ADMITTED BEFORE 2022

cursor.execute("""
UPDATE marks
SET marks_obtained = marks_obtained + 5
WHERE student_id IN (
    SELECT student_id
    FROM students
    WHERE admission_year < 2022
)
""")

conn.commit()

# STEP 8: FIND STUDENTS WHO FAILED IN MORE THAN 2 SUBJECTS

failed_query = """
SELECT student_id
FROM marks
WHERE marks_obtained < 35
GROUP BY student_id
HAVING COUNT(*) > 2
"""

failed_students = [
    row[0] for row in cursor.execute(failed_query).fetchall()
]

# STEP 9: DELETE THEIR MARKS AND STUDENT RECORDS

cursor.execute("""
DELETE FROM marks
WHERE student_id IN (
    SELECT student_id
    FROM (
        SELECT student_id
        FROM marks
        WHERE marks_obtained < 35
        GROUP BY student_id
        HAVING COUNT(*) > 2
    )
)
""")

cursor.execute("""
DELETE FROM students
WHERE student_id IN (
    SELECT student_id
    FROM (
        SELECT student_id
        FROM marks
        WHERE marks_obtained < 35
        GROUP BY student_id
        HAVING COUNT(*) > 2
    )
)
""")

conn.commit()

# STEP 10: DEPARTMENT-WISE AVERAGE MARKS AND STUDENT COUNT

department_query = """
SELECT
    s.department,
    ROUND(AVG(m.marks_obtained), 2) AS average_marks,
    COUNT(DISTINCT s.student_id) AS number_of_students
FROM students s
JOIN marks m
ON s.student_id = m.student_id
GROUP BY s.department
"""

department_df = pd.read_sql_query(department_query, conn)
print("Department-wise Analysis:")
display(department_df)

# STEP 11: CREATE VIEW FOR AVERAGE MARKS ABOVE 75

cursor.execute("""
DROP VIEW IF EXISTS high_performing_students
""")

cursor.execute("""
CREATE VIEW high_performing_students AS
SELECT
    s.student_id,
    s.student_name,
    s.department,
    ROUND(AVG(m.marks_obtained), 2) AS average_marks
FROM students s
JOIN marks m
ON s.student_id = m.student_id
GROUP BY
    s.student_id,
    s.student_name,
    s.department
HAVING AVG(m.marks_obtained) > 75
""")

conn.commit()

# STEP 12: DISPLAY HIGH-PERFORMING STUDENTS

high_performers = pd.read_sql_query(
    "SELECT * FROM high_performing_students", conn
)

print("High-Performing Students:")
display(high_performers)

# STEP 13: DISPLAY FINAL STUDENT TABLE

print("Final Students Table:")
display(pd.read_sql_query("SELECT * FROM students", conn))

# STEP 14: DISPLAY FINAL MARKS TABLE

print("Final Marks Table:")
display(pd.read_sql_query("SELECT * FROM marks", conn))

# STEP 15: CLOSE DATABASE CONNECTION

conn.close()

print("Academic Records Project Completed Successfully!")

Department-wise Analysis:


,department,average_marks,number_of_students
0,CSE,87.33,4
1,ECE,59.22,3
2,IT,67.00,3


High-Performing Students:


,student_id,student_name,department,average_marks
0,1,Arun,CSE,83.33
1,2,Priya,CSE,91.67
2,4,Divya,ECE,77.67
3,6,Meena,IT,92.67
4,7,Anu,CSE,94.33
5,10,Siva,CSE,80.00


Final Students Table:


,student_id,student_name,department,admission_year
0,1,Arun,CSE,2020
1,2,Priya,CSE,2021
2,3,Kumar,ECE,2022
3,4,Divya,ECE,2023
4,5,Ravi,IT,2020
5,6,Meena,IT,2021
6,7,Anu,CSE,2024
7,8,Vijay,ECE,2019
8,9,Kavi,IT,2022
9,10,Siva,CSE,2023


Final Marks Table:


,mark_id,student_id,subject,marks_obtained
0,1,1,Python,85
1,2,1,SQL,75
2,3,1,Maths,90
3,4,2,Python,95
4,5,2,SQL,87
5,6,2,Maths,93
6,7,3,Python,60
7,8,3,SQL,65
8,9,3,Maths,70
9,10,4,Python,78


Academic Records Project Completed Successfully!


In [4]:
# PROBLEM STATEMENT 2: EMPLOYEE MANAGEMENT & SALARY CONTROL

# STEP 1: IMPORT LIBRARIES

import sqlite3
import pandas as pd

# STEP 2: CONNECT TO DATABASE

conn = sqlite3.connect("employee_management.db")
conn.execute("PRAGMA foreign_keys = ON")
cursor = conn.cursor()

# STEP 3: CREATE DEPARTMENTS TABLE

cursor.execute("""
CREATE TABLE IF NOT EXISTS departments (
    dept_id INTEGER PRIMARY KEY,
    dept_name TEXT UNIQUE
)
""")

# STEP 4: CREATE EMPLOYEES TABLE

cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    emp_id INTEGER PRIMARY KEY,
    emp_name TEXT NOT NULL,
    dept_id INTEGER,
    salary REAL,
    joining_date TEXT,
    FOREIGN KEY (dept_id)
        REFERENCES departments(dept_id)
)
""")

# STEP 5: INSERT DEPARTMENTS

departments_data = [
    (1, 'IT'),
    (2, 'HR'),
    (3, 'Finance'),
    (4, 'Sales'),
    (5, 'Marketing'),
    (6, 'Research')
]

cursor.executemany("""
INSERT OR IGNORE INTO departments
(dept_id, dept_name)
VALUES (?, ?)
""", departments_data)

# STEP 6: INSERT EMPLOYEE DETAILS

employees_data = [
    (101, 'Arun', 1, 95000, '2018-05-10'),
    (102, 'Priya', 1, 85000, '2019-03-15'),
    (103, 'Kumar', 2, 60000, '2023-06-01'),
    (104, 'Divya', 2, 70000, '2017-08-20'),
    (105, 'Ravi', 3, 90000, '2016-01-12'),
    (106, 'Meena', 3, 75000, '2022-09-05'),
    (107, 'Anu', 4, 50000, '2024-02-10'),
    (108, 'Vijay', 4, 65000, '2018-11-25'),
    (109, 'Kavi', 5, 80000, '2020-04-18'),
    (110, 'Siva', 1, 110000, '2015-07-07')
]

cursor.executemany("""
INSERT OR IGNORE INTO employees
(emp_id, emp_name, dept_id, salary, joining_date)
VALUES (?, ?, ?, ?, ?)
""", employees_data)

conn.commit()

# STEP 7: DISPLAY INITIAL EMPLOYEE DETAILS

print("Initial Employee Details:")

display(pd.read_sql_query("""
SELECT
    e.emp_id,
    e.emp_name,
    d.dept_name,
    e.salary,
    e.joining_date
FROM employees e
JOIN departments d
ON e.dept_id = d.dept_id
ORDER BY e.emp_id
""", conn))

# STEP 8: INCREASE SALARY BY 10% FOR EMPLOYEES
# WITH MORE THAN 5 YEARS OF EXPERIENCE
# AS OF 2026-10-29

cursor.execute("""
UPDATE employees
SET salary = salary * 1.10
WHERE date(joining_date) <
      date('2026-10-29', '-5 years')
""")

conn.commit()

# STEP 9: DECREASE SALARY BY 5% IN DEPARTMENTS
# WHERE AVERAGE SALARY EXCEEDS 80000
# USING SALARIES AFTER THE PREVIOUS UPDATE

cursor.execute("""
UPDATE employees
SET salary = salary * 0.95
WHERE dept_id IN (
    SELECT dept_id
    FROM employees
    GROUP BY dept_id
    HAVING AVG(salary) > 80000
)
""")

conn.commit()

# STEP 10: DELETE DEPARTMENTS WITH NO EMPLOYEES

cursor.execute("""
DELETE FROM departments
WHERE NOT EXISTS (
    SELECT 1
    FROM employees
    WHERE employees.dept_id = departments.dept_id
)
""")

conn.commit()

# STEP 11: DEPARTMENT-WISE TOTAL SALARY COST

print("Department-wise Total Salary Cost:")

display(pd.read_sql_query("""
SELECT
    d.dept_name,
    ROUND(SUM(e.salary), 2) AS total_salary_cost
FROM departments d
JOIN employees e
ON d.dept_id = e.dept_id
GROUP BY d.dept_id, d.dept_name
ORDER BY total_salary_cost DESC
""", conn))

# STEP 12: DEPARTMENT-WISE HIGHEST PAID EMPLOYEE

print("Highest Paid Employee in Each Department:")

display(pd.read_sql_query("""
SELECT
    d.dept_name,
    e.emp_name,
    e.salary
FROM employees e
JOIN departments d
ON e.dept_id = d.dept_id
WHERE e.salary = (
    SELECT MAX(e2.salary)
    FROM employees e2
    WHERE e2.dept_id = e.dept_id
)
ORDER BY d.dept_name
""", conn))

# STEP 13: FINAL EMPLOYEE TABLE

print("Final Employee Details:")

display(pd.read_sql_query("""
SELECT
    e.emp_id,
    e.emp_name,
    d.dept_name,
    e.salary,
    e.joining_date
FROM employees e
JOIN departments d
ON e.dept_id = d.dept_id
ORDER BY e.emp_id
""", conn))

# STEP 14: FINAL DEPARTMENTS TABLE

print("Final Departments:")

display(pd.read_sql_query("""
SELECT * FROM departments
""", conn))

# STEP 15: CLOSE DATABASE CONNECTION

conn.close()

print("Employee Management Project Completed Successfully!")

Initial Employee Details:


,emp_id,emp_name,dept_name,salary,joining_date
0,101,Arun,IT,99275.0,2018-05-10
1,102,Priya,IT,88825.0,2019-03-15
2,103,Kumar,HR,60000.0,2023-06-01
3,104,Divya,HR,77000.0,2017-08-20
4,105,Ravi,Finance,94050.0,2016-01-12
5,106,Meena,Finance,71250.0,2022-09-05
6,107,Anu,Sales,50000.0,2024-02-10
7,108,Vijay,Sales,71500.0,2018-11-25
8,109,Kavi,Marketing,83600.0,2020-04-18
9,110,Siva,IT,114950.0,2015-07-07


Department-wise Total Salary Cost:


,dept_name,total_salary_cost
0,IT,316687.25
1,Finance,165969.75
2,HR,144700.00
3,Sales,128650.00
4,Marketing,87362.00


Highest Paid Employee in Each Department:


...

Final Employee Details:


,emp_id,emp_name,dept_name,salary,joining_date
0,101,Arun,IT,103742.375,2018-05-10
1,102,Priya,IT,92822.125,2019-03-15
2,103,Kumar,HR,60000.000,2023-06-01
3,104,Divya,HR,84700.000,2017-08-20
4,105,Ravi,Finance,98282.250,2016-01-12
5,106,Meena,Finance,67687.500,2022-09-05
6,107,Anu,Sales,50000.000,2024-02-10
7,108,Vijay,Sales,78650.000,2018-11-25
8,109,Kavi,Marketing,87362.000,2020-04-18
9,110,Siva,IT,120122.750,2015-07-07


Final Departments:


,dept_id,dept_name
0,1,IT
1,2,HR
2,3,Finance
3,4,Sales
4,5,Marketing


Employee Management Project Completed Successfully!


In [6]:
# Problem Statement 4 — Inventory Optimization & Data Cleanup
# STEP 1: IMPORT LIBRARY
import sqlite3
import pandas as pd

# STEP 2: CREATE DATABASE CONNECTION
conn = sqlite3.connect("inventory.db")
cursor = conn.cursor()

# STEP 3: CREATE SUPPLIERS TABLE
cursor.execute("""
CREATE TABLE IF NOT EXISTS suppliers (
    supplier_id INTEGER PRIMARY KEY,
    supplier_name TEXT NOT NULL,
    rating INTEGER
)
""")

# STEP 4: CREATE PRODUCTS TABLE
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT NOT NULL,
    supplier_id INTEGER,
    price REAL,
    stock INTEGER,
    FOREIGN KEY (supplier_id)
        REFERENCES suppliers(supplier_id)
)
""")

# STEP 5: INSERT SUPPLIER DETAILS
cursor.executemany("""
INSERT OR REPLACE INTO suppliers
(supplier_id, supplier_name, rating)
VALUES (?, ?, ?)
""", [
    (1, 'ABC Suppliers', 5),
    (2, 'XYZ Traders', 2),
    (3, 'Global Stores', 4),
    (4, 'Prime Distributors', 1)
])

# STEP 6: INSERT PRODUCT DETAILS
cursor.executemany("""
INSERT OR REPLACE INTO products
(product_id, product_name, supplier_id, price, stock)
VALUES (?, ?, ?, ?, ?)
""", [
    (101, 'Laptop', 1, 80000, 10),
    (102, 'Keyboard', 1, 1500, 50),
    (103, 'Monitor', 2, 12000, 25),
    (104, 'Printer', 2, 25000, 0),
    (105, 'Server', 3, 150000, 5),
    (106, 'Mouse', 3, 800, 15),
    (107, 'Scanner', 4, 10000, 30),
    (108, 'Tablet', 1, 30000, 18)
])

conn.commit()

# STEP 7: DISPLAY SUPPLIERS
print("SUPPLIERS TABLE")
display(pd.read_sql_query(
    "SELECT * FROM suppliers", conn
))

# STEP 8: DISPLAY PRODUCTS
print("PRODUCTS TABLE")
display(pd.read_sql_query(
    "SELECT * FROM products", conn
))

# STEP 9: INCREASE PRICE BY 15%
cursor.execute("""
UPDATE products
SET price = price * 1.15
WHERE stock < 20
AND supplier_id IN (
    SELECT supplier_id
    FROM suppliers
    WHERE rating >= 4
)
""")
conn.commit()

# STEP 10: DECREASE PRICE BY 10%
cursor.execute("""
UPDATE products
SET price = price * 0.90
WHERE supplier_id IN (
    SELECT supplier_id
    FROM suppliers
    WHERE rating <= 2
)
""")
conn.commit()

# STEP 11: DISPLAY UPDATED PRODUCTS
print("UPDATED PRODUCT PRICES")
display(pd.read_sql_query(
    "SELECT * FROM products", conn
))

# STEP 12: DELETE UNWANTED PRODUCTS
cursor.execute("""
DELETE FROM products
WHERE stock = 0
OR price > 100000
""")
conn.commit()

# STEP 13: DISPLAY CLEANED PRODUCTS
print("PRODUCTS AFTER DATA CLEANING")
display(pd.read_sql_query(
    "SELECT * FROM products", conn
))

# STEP 14: SUPPLIER-WISE TOTAL STOCK
print("SUPPLIER-WISE TOTAL STOCK")
display(pd.read_sql_query("""
SELECT
    s.supplier_name,
    SUM(p.stock) AS total_stock
FROM suppliers s
JOIN products p
ON s.supplier_id = p.supplier_id
GROUP BY s.supplier_id, s.supplier_name
""", conn))

# STEP 15: SUPPLIER-WISE AVERAGE PRICE
print("SUPPLIER-WISE AVERAGE PRICE")
display(pd.read_sql_query("""
SELECT
    s.supplier_name,
    ROUND(AVG(p.price), 2) AS average_price
FROM suppliers s
JOIN products p
ON s.supplier_id = p.supplier_id
GROUP BY s.supplier_id, s.supplier_name
""", conn))

# STEP 16: TOTAL STOCK AND AVERAGE PRICE
print("FINAL INVENTORY REPORT")
display(pd.read_sql_query("""
SELECT
    s.supplier_id,
    s.supplier_name,
    SUM(p.stock) AS total_stock,
    ROUND(AVG(p.price), 2) AS average_price
FROM suppliers s
JOIN products p
ON s.supplier_id = p.supplier_id
GROUP BY s.supplier_id, s.supplier_name
""", conn))

# STEP 17: CLOSE DATABASE
conn.close()

print("Inventory Optimization Project Completed!")



SUPPLIERS TABLE


,supplier_id,supplier_name,rating
0,1,ABC Suppliers,5
1,2,XYZ Traders,2
2,3,Global Stores,4
3,4,Prime Distributors,1


PRODUCTS TABLE


,product_id,product_name,supplier_id,price,stock
0,101,Laptop,1,80000.0,10
1,102,Keyboard,1,1500.0,50
2,103,Monitor,2,12000.0,25
3,104,Printer,2,25000.0,0
4,105,Server,3,150000.0,5
5,106,Mouse,3,800.0,15
6,107,Scanner,4,10000.0,30
7,108,Tablet,1,30000.0,18


UPDATED PRODUCT PRICES


...

SUPPLIER-WISE AVERAGE PRICE


,supplier_name,average_price
0,ABC Suppliers,42666.67
1,XYZ Traders,10800.00
2,Global Stores,920.00
3,Prime Distributors,9000.00


FINAL INVENTORY REPORT


,supplier_id,supplier_name,total_stock,average_price
0,1,ABC Suppliers,78,42666.67
1,2,XYZ Traders,25,10800.00
2,3,Global Stores,15,920.00
3,4,Prime Distributors,30,9000.00


Inventory Optimization Project Completed!


In [8]:
# PROBLEM STATEMENT 5 — Banking Transactions & Fraud Detection
# STEP 1: IMPORT LIBRARIES
import sqlite3
import pandas as pd

# STEP 2: CREATE DATABASE CONNECTION
conn = sqlite3.connect("banking.db")
cursor = conn.cursor()

# STEP 3: CREATE ACCOUNTS TABLE
cursor.execute("""
CREATE TABLE IF NOT EXISTS accounts (
    account_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    account_type TEXT,
    balance REAL
)
""")

# STEP 4: CREATE TRANSACTIONS TABLE
cursor.execute("""
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id INTEGER PRIMARY KEY,
    account_id INTEGER,
    transaction_date TEXT,
    amount REAL,
    transaction_type TEXT
        CHECK (transaction_type IN ('Debit', 'Credit')),
    FOREIGN KEY (account_id)
        REFERENCES accounts(account_id)
)
""")

# STEP 5: INSERT ACCOUNT DETAILS
cursor.executemany("""
INSERT OR REPLACE INTO accounts
(account_id, customer_name, account_type, balance)
VALUES (?, ?, ?, ?)
""", [
    (101, 'Arun', 'Savings', 5000),
    (102, 'Priya', 'Current', 10000),
    (103, 'Kumar', 'Savings', 800),
    (104, 'Divya', 'Current', 20000),
    (105, 'Ravi', 'Savings', 3000)
])

# STEP 6: INSERT TRANSACTION DETAILS
cursor.executemany("""
INSERT OR REPLACE INTO transactions
(transaction_id, account_id, transaction_date,
 amount, transaction_type)
VALUES (?, ?, ?, ?, ?)
""", [
    (1, 101, '2026-10-20', 1000, 'Debit'),
    (2, 101, '2026-10-21', 2000, 'Credit'),
    (3, 101, '2026-10-22', 500, 'Debit'),
    (4, 102, '2026-10-20', 3000, 'Debit'),
    (5, 102, '2026-10-21', 1000, 'Credit'),
    (6, 103, '2026-10-22', 500, 'Debit'),
    (7, 103, '2026-10-23', 200, 'Credit'),
    (8, 104, '2026-10-23', 5000, 'Credit'),
    (9, 104, '2026-10-24', 2000, 'Debit'),
    (10, 105, '2026-10-24', 1000, 'Debit'),
    (11, 105, '2026-10-25', 500, 'Credit')
])

conn.commit()

# STEP 7: DISPLAY ACCOUNTS
print("ACCOUNTS TABLE")
display(pd.read_sql_query(
    "SELECT * FROM accounts", conn
))

# STEP 8: DISPLAY TRANSACTIONS
print("TRANSACTIONS TABLE")
display(pd.read_sql_query(
    "SELECT * FROM transactions", conn
))

# STEP 9: UPDATE ACCOUNT BALANCES
# Debit reduces balance; Credit increases balance.
# Apply transactions only once to avoid double counting.

cursor.execute("""
UPDATE accounts
SET balance = balance
    + COALESCE((
        SELECT SUM(t.amount)
        FROM transactions t
        WHERE t.account_id = accounts.account_id
          AND t.transaction_type = 'Credit'
    ), 0)
    - COALESCE((
        SELECT SUM(t.amount)
        FROM transactions t
        WHERE t.account_id = accounts.account_id
          AND t.transaction_type = 'Debit'
    ), 0)
""")

conn.commit()

# STEP 10: DISPLAY UPDATED BALANCES
print("UPDATED ACCOUNT BALANCES")
display(pd.read_sql_query("""
SELECT account_id, customer_name, account_type, balance
FROM accounts
""", conn))

# STEP 11: IDENTIFY ACCOUNTS WHERE
# TOTAL DEBIT EXCEEDS TOTAL CREDIT

print("RISKY ACCOUNTS: DEBIT > CREDIT")

display(pd.read_sql_query("""
SELECT
    a.account_id,
    a.customer_name,
    COALESCE(SUM(
        CASE WHEN t.transaction_type = 'Debit'
             THEN t.amount ELSE 0 END
    ), 0) AS total_debit,
    COALESCE(SUM(
        CASE WHEN t.transaction_type = 'Credit'
             THEN t.amount ELSE 0 END
    ), 0) AS total_credit
FROM accounts a
LEFT JOIN transactions t
    ON a.account_id = t.account_id
GROUP BY a.account_id, a.customer_name
HAVING
    COALESCE(SUM(
        CASE WHEN t.transaction_type = 'Debit'
             THEN t.amount ELSE 0 END
    ), 0)
    >
    COALESCE(SUM(
        CASE WHEN t.transaction_type = 'Credit'
             THEN t.amount ELSE 0 END
    ), 0)
""", conn))

# STEP 12: FLAG LOW-BALANCE ACCOUNTS USING CASE

print("LOW BALANCE ALERT")

display(pd.read_sql_query("""
SELECT
    account_id,
    customer_name,
    balance,
    CASE
        WHEN balance < 1000 THEN 'LOW BALANCE - FLAG'
        ELSE 'NORMAL'
    END AS account_status
FROM accounts
""", conn))

# STEP 13: ACCOUNT TYPE-WISE TRANSACTION REPORT

print("ACCOUNT TYPE-WISE REPORT")

display(pd.read_sql_query("""
SELECT
    a.account_type,
    COUNT(t.transaction_id) AS total_transactions,
    ROUND(AVG(a.balance), 2) AS average_balance
FROM accounts a
LEFT JOIN transactions t
    ON a.account_id = t.account_id
GROUP BY a.account_type
""", conn))

# STEP 14: COMPLETE ACCOUNT TRANSACTION SUMMARY

print("COMPLETE BANKING REPORT")

display(pd.read_sql_query("""
SELECT
    a.account_id,
    a.customer_name,
    a.account_type,
    a.balance,
    COUNT(t.transaction_id) AS total_transactions,
    COALESCE(SUM(
        CASE WHEN t.transaction_type = 'Debit'
             THEN t.amount ELSE 0 END
    ), 0) AS total_debit,
    COALESCE(SUM(
        CASE WHEN t.transaction_type = 'Credit'
             THEN t.amount ELSE 0 END
    ), 0) AS total_credit,
    CASE
        WHEN a.balance < 1000 THEN 'RISKY'
        ELSE 'NORMAL'
    END AS risk_status
FROM accounts a
LEFT JOIN transactions t
    ON a.account_id = t.account_id
GROUP BY
    a.account_id,
    a.customer_name,
    a.account_type,
    a.balance
""", conn))

# STEP 15: CLOSE DATABASE
conn.close()

print("Banking Transactions & Fraud Detection Project Completed!")



ACCOUNTS TABLE


,account_id,customer_name,account_type,balance
0,101,Arun,Savings,5000.0
1,102,Priya,Current,10000.0
2,103,Kumar,Savings,800.0
3,104,Divya,Current,20000.0
4,105,Ravi,Savings,3000.0


TRANSACTIONS TABLE


,transaction_id,account_id,transaction_date,amount,transaction_type
0,1,101,2026-10-20,1000.0,Debit
1,2,101,2026-10-21,2000.0,Credit
2,3,101,2026-10-22,500.0,Debit
3,4,102,2026-10-20,3000.0,Debit
4,5,102,2026-10-21,1000.0,Credit
5,6,103,2026-10-22,500.0,Debit
6,7,103,2026-10-23,200.0,Credit
7,8,104,2026-10-23,5000.0,Credit
8,9,104,2026-10-24,2000.0,Debit
9,10,105,2026-10-24,1000.0,Debit


UPDATED ACCOUNT BALANCES


...

ACCOUNT TYPE-WISE REPORT


,account_type,total_transactions,average_balance
0,Current,4,15500.00
1,Savings,7,3214.29


COMPLETE BANKING REPORT


,account_id,customer_name,account_type,balance,total_transactions,total_debit,total_credit,risk_status
0,101,Arun,Savings,5500.0,3,1500.0,2000.0,NORMAL
1,102,Priya,Current,8000.0,2,3000.0,1000.0,NORMAL
2,103,Kumar,Savings,500.0,2,500.0,200.0,RISKY
3,104,Divya,Current,23000.0,2,2000.0,5000.0,NORMAL
4,105,Ravi,Savings,2500.0,2,1000.0,500.0,NORMAL


Banking Transactions & Fraud Detection Project Completed!


In [10]:
# PROBLEM STATEMENT 3 — Sales System with Revenue & Risk Analysis
# STEP 1: IMPORT LIBRARIES
import sqlite3
import pandas as pd

# STEP 2: CREATE DATABASE CONNECTION
conn = sqlite3.connect("sales.db")
cursor = conn.cursor()

# STEP 3: CREATE CUSTOMERS TABLE
cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id INTEGER PRIMARY KEY,
    customer_name TEXT NOT NULL,
    city TEXT
)
""")

# STEP 4: CREATE ORDERS TABLE
cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    order_amount REAL,
    payment_mode TEXT,
    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
)
""")

# STEP 5: INSERT CUSTOMER DETAILS
cursor.executemany("""
INSERT OR REPLACE INTO customers
(customer_id, customer_name, city)
VALUES (?, ?, ?)
""", [
    (1, 'Arun', 'Chennai'),
    (2, 'Priya', 'Coimbatore'),
    (3, 'Kumar', 'Trichy'),
    (4, 'Divya', 'Chennai'),
    (5, 'Ravi', 'Madurai')
])

# STEP 6: INSERT SALES DATA
cursor.executemany("""
INSERT OR REPLACE INTO orders
(order_id, customer_id, order_date,
 order_amount, payment_mode)
VALUES (?, ?, ?, ?, ?)
""", [
    (101, 1, '2026-05-10', 60000, 'UPI'),
    (102, 1, '2026-06-15', 50000, 'UPI'),
    (103, 2, '2026-07-20', 40000, 'UPI'),
    (104, 2, '2026-08-10', 20000, 'Cash'),
    (105, 3, '2022-01-10', 3000, 'Cash'),
    (106, 3, '2026-03-12', 15000, 'Card'),
    (107, 4, '2021-05-15', 4000, 'UPI'),
    (108, 4, '2026-09-01', 80000, 'UPI'),
    (109, 5, '2026-02-18', 25000, 'Card'),
    (110, 5, '2023-06-20', 4500, 'Cash')
])

conn.commit()

# STEP 7: DISPLAY CUSTOMERS
print("CUSTOMERS TABLE")
display(pd.read_sql_query(
    "SELECT * FROM customers", conn
))

# STEP 8: DISPLAY ORDERS
print("ORDERS TABLE")
display(pd.read_sql_query(
    "SELECT * FROM orders", conn
))

# STEP 9: IDENTIFY CUSTOMERS WHOSE
# TOTAL PURCHASE EXCEEDS 100000

print("LOYAL CUSTOMERS: TOTAL PURCHASE > 100000")

display(pd.read_sql_query("""
SELECT
    c.customer_id,
    c.customer_name,
    SUM(o.order_amount) AS total_purchase
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name
HAVING SUM(o.order_amount) > 100000
""", conn))

# STEP 10: APPLY 5% DISCOUNT TO UPI ORDERS
# OF LOYAL CUSTOMERS
# Loyalty = total purchase > 100000 before discount

cursor.execute("""
UPDATE orders
SET order_amount = order_amount * 0.95
WHERE payment_mode = 'UPI'
AND customer_id IN (
    SELECT customer_id
    FROM orders
    GROUP BY customer_id
    HAVING SUM(order_amount) > 100000
)
""")

conn.commit()

# STEP 11: DISPLAY ORDERS AFTER DISCOUNT
print("ORDERS AFTER 5% UPI DISCOUNT")

display(pd.read_sql_query("""
SELECT * FROM orders
""", conn))

# STEP 12: DELETE OLD, LOW-VALUE ORDERS
# Older than 3 years AND amount < 5000
# Reference date: 2026-10-29

cursor.execute("""
DELETE FROM orders
WHERE order_date < date('2026-10-29', '-3 years')
AND order_amount < 5000
""")

conn.commit()

# STEP 13: DISPLAY ORDERS AFTER CLEANUP
print("ORDERS AFTER DATA CLEANING")

display(pd.read_sql_query("""
SELECT * FROM orders
""", conn))

# STEP 14: CITY-WISE TOTAL REVENUE
# AND NUMBER OF ORDERS

print("CITY-WISE SALES REPORT")

display(pd.read_sql_query("""
SELECT
    c.city,
    ROUND(SUM(o.order_amount), 2) AS total_revenue,
    COUNT(o.order_id) AS number_of_orders
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY c.city
ORDER BY total_revenue DESC
""", conn))

# STEP 15: COMPLETE CUSTOMER SALES REPORT

print("CUSTOMER-WISE SALES SUMMARY")

display(pd.read_sql_query("""
SELECT
    c.customer_id,
    c.customer_name,
    c.city,
    COUNT(o.order_id) AS total_orders,
    COALESCE(SUM(o.order_amount), 0) AS total_purchase
FROM customers c
LEFT JOIN orders o
    ON c.customer_id = o.customer_id
GROUP BY
    c.customer_id,
    c.customer_name,
    c.city
ORDER BY total_purchase DESC
""", conn))

# STEP 16: CLOSE DATABASE
conn.close()

print("Sales System Project Completed Successfully!")



CUSTOMERS TABLE


,customer_id,customer_name,city
0,1,Arun,Chennai
1,2,Priya,Coimbatore
2,3,Kumar,Trichy
3,4,Divya,Chennai
4,5,Ravi,Madurai


ORDERS TABLE


,order_id,customer_id,order_date,order_amount,payment_mode
0,101,1,2026-05-10,60000.0,UPI
1,102,1,2026-06-15,50000.0,UPI
2,103,2,2026-07-20,40000.0,UPI
3,104,2,2026-08-10,20000.0,Cash
4,105,3,2022-01-10,3000.0,Cash
5,106,3,2026-03-12,15000.0,Card
6,107,4,2021-05-15,4000.0,UPI
7,108,4,2026-09-01,80000.0,UPI
8,109,5,2026-02-18,25000.0,Card
9,110,5,2023-06-20,4500.0,Cash


LOYAL CUSTOMERS: TOTAL PURCHASE > 100000


...

CITY-WISE SALES REPORT


,city,total_revenue,number_of_orders
0,Chennai,184500.0,3
1,Coimbatore,60000.0,2
2,Madurai,25000.0,1
3,Trichy,15000.0,1


CUSTOMER-WISE SALES SUMMARY


,customer_id,customer_name,city,total_orders,total_purchase
0,1,Arun,Chennai,2,104500.0
1,4,Divya,Chennai,1,80000.0
2,2,Priya,Coimbatore,2,60000.0
3,5,Ravi,Madurai,1,25000.0
4,3,Kumar,Trichy,1,15000.0


Sales System Project Completed Successfully!
